In [1]:
# !pip install -U sentence-transformers langchain-huggingface

In [1]:
import os
import langchain
import warnings
warnings.filterwarnings('ignore')
os.environ["HUGGINGFACEHUB_API_TOKEN"] = "" #  put your id

In [2]:
from langchain_community.document_loaders import PyPDFLoader  # it is used to read the pdf

In [3]:
#  step 1  connect to LLM in this case--> Connect to DeepSeek-V3
from langchain_huggingface import HuggingFaceEndpoint ,  ChatHuggingFace
llm = HuggingFaceEndpoint(
    repo_id="deepseek-ai/DeepSeek-V3",    #  repo_id is used to tell which model to be used  LLM  model
    max_new_tokens= 512,                   # max number of words at a time
    temperature= .2,                      #  more  temp more creativity, less temp more reliablity
)
chat_model = ChatHuggingFace(llm=llm)

print("✅ DeepSeek-V3 LLM connected successfully!")

✅ DeepSeek-V3 LLM connected successfully!


In [4]:
import os
from langchain_community.document_loaders import PyPDFDirectoryLoader

# All Loan / Housing Finance PDF documents
pdf_folder = r"C:\Users\amoln\OneDrive\Documents\Loan_Guidelines(project)\Documents"

# Load all PDF files from the folder
loader = PyPDFDirectoryLoader(pdf_folder)
docs = loader.load()
print("Total pages loaded:", len(docs))

# List all PDF files in the folder
pdf_files = [
    file for file in os.listdir(pdf_folder)
    if file.lower().endswith(".pdf")
]

print("\nPDF files in the folder:")

for i, file in enumerate(pdf_files, start=1):
    print(f"{i}. {file}")

print(f"\nTotal PDF files: {len(pdf_files)}")
print(f"Total pages/documents loaded: {len(docs)}")

Total pages loaded: 568

PDF files in the folder:
1. ICICI_Home_Loan_Documents.pdf
2. ICICI_Home_Loan_FAQs.pdf
3. Ombudsman_Scheme_English.pdf
4. Operational-Guidelines-of-PMAY-U-2-Hindi.pdf
5. Operational-Guidelines-of-PMAY-U-2.pdf
6. Operational-Guidelines-of-PMAY-U.pdf
7. RBI-charges.pdf
8. RBI-Intresti-on-loans.pdf
9. RBI-keywords.pdf
10. RBi-penalty-charges.pdf
11. RBI-Resposible-docs.pdf
12. RBI-setlement.pdf
13. RBI_mastery.pdf
14. SBI_HOME_LOANS_APPLICATION_FORM.pdf
15. SBI_HOME_LOAN_MITC.pdf

Total PDF files: 15
Total pages/documents loaded: 568


In [5]:
from collections import Counter
page_counts = Counter(doc.metadata['source'] for doc in docs)
for src, count in page_counts.items():
    print(f"{src} -> {count} pages")

C:\Users\amoln\OneDrive\Documents\Loan_Guidelines(project)\Documents\ICICI_Home_Loan_Documents.pdf -> 5 pages
C:\Users\amoln\OneDrive\Documents\Loan_Guidelines(project)\Documents\ICICI_Home_Loan_FAQs.pdf -> 13 pages
C:\Users\amoln\OneDrive\Documents\Loan_Guidelines(project)\Documents\Ombudsman_Scheme_English.pdf -> 19 pages
C:\Users\amoln\OneDrive\Documents\Loan_Guidelines(project)\Documents\Operational-Guidelines-of-PMAY-U-2-Hindi.pdf -> 112 pages
C:\Users\amoln\OneDrive\Documents\Loan_Guidelines(project)\Documents\Operational-Guidelines-of-PMAY-U-2.pdf -> 112 pages
C:\Users\amoln\OneDrive\Documents\Loan_Guidelines(project)\Documents\Operational-Guidelines-of-PMAY-U.pdf -> 94 pages
C:\Users\amoln\OneDrive\Documents\Loan_Guidelines(project)\Documents\RBI-charges.pdf -> 44 pages
C:\Users\amoln\OneDrive\Documents\Loan_Guidelines(project)\Documents\RBI-Intresti-on-loans.pdf -> 3 pages
C:\Users\amoln\OneDrive\Documents\Loan_Guidelines(project)\Documents\RBI-keywords.pdf -> 44 pages
C:\User

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# 1. Split documents into chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.split_documents(docs)

print("Total chunks:", len(chunks))

Total chunks: 1598


In [7]:
print("Total chunks:", len(chunks))

for i, chunk in enumerate(chunks[:5]):
    print("=" * 80)
    print("CHUNK:", i + 1)
    print("Source:", os.path.basename(chunk.metadata.get("source", "")))
    print("Page:", chunk.metadata.get("page"))
    print("Text length:", len(chunk.page_content))
    print("Text:", chunk.page_content[:700])
    print()

Total chunks: 1598
CHUNK: 1
Source: ICICI_Home_Loan_Documents.pdf
Page: 0
Text length: 973
Text: ICICI Bank - ICICI Home Loan Documents
Official Source: https://www.icici.bank.in/personal-banking/loans/home-loan/documents-required
List of Documents Required for Home Loan in 2026 | ICICI Bank
Home Loan
Documents Required For Home Loan
APPLY NOW
Loans
Home Loan
Home Loan Documents
Documents Required For Home Loan
Getting a
Home Loan
from ICICI Bank is easy and quick. We know you may need a Home Loan for the purpose of buying or constructing a new
home and in some cases to renovate the house you currently live in. Whatever may be the reason, we have minimum
paperwork and other formalities to avail a Home Loan. The list of documents required for Home Loan are proof of your identity,
address

CHUNK: 2
Source: ICICI_Home_Loan_Documents.pdf
Page: 0
Text length: 968
Text: APPLY FOR HOME LOAN
List of Documents required for Home Loan Based on profile
Individuals:
Identity proof and address proof

In [8]:
# 2. Multilingual embedding model
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

print("✅ Multilingual embedding model loaded!")

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Multilingual embedding model loaded!


In [9]:
from sentence_transformers import SentenceTransformer

In [10]:
test_embedding = embeddings.embed_query(
    "होम लोनसाठी कोणती कागदपत्रे आवश्यक आहेत?"
)

print("Embedding dimension:", len(test_embedding))

Embedding dimension: 384


In [11]:
# 3. Create FAISS vector database
vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

print("✅ FAISS vector store created!")
print("Total vectors:", vectorstore.index.ntotal)

✅ FAISS vector store created!
Total vectors: 1598


In [12]:
# 4. Create retriever
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 6}
)

print("✅ Retriever ready!")

✅ Retriever ready!


In [13]:
query = "होम लोनसाठी कोणती कागदपत्रे आवश्यक आहेत?"

results = retriever.invoke(query)

for i, doc in enumerate(results, 1):

    print("=" * 80)
    print("RESULT:", i)
    print("Source:", os.path.basename(doc.metadata.get("source", "")))
    print("Page:", doc.metadata.get("page"))
    print(doc.page_content[:500])

RESULT: 1
Source: ICICI_Home_Loan_Documents.pdf
Page: 0
APPLY FOR HOME LOAN
List of Documents required for Home Loan Based on profile
Individuals:
Identity proof and address proof of individuals:
Aadhaar or Passport or Driving License or Voter ID or National Rural Employment Guarantee Act (NREGA) Card or Letter from
the National Population Register (NPR) containing details of your name and address.
DOB Proof:
Passport/Driving License/Voter ID/NREGA Card/Letter from the NPR containing details of your name and address.
Non-Individuals (Entity):
Proof o
RESULT: 2
Source: ICICI_Home_Loan_Documents.pdf
Page: 0
ICICI Bank - ICICI Home Loan Documents
Official Source: https://www.icici.bank.in/personal-banking/loans/home-loan/documents-required
List of Documents Required for Home Loan in 2026 | ICICI Bank
Home Loan
Documents Required For Home Loan
APPLY NOW
Loans
Home Loan
Home Loan Documents
Documents Required For Home Loan
Getting a
Home Loan
from ICICI Bank is easy and quick. We know you ma

In [14]:
query = "What documents are required for a home loan?"

results = retriever.invoke(query)

for i, doc in enumerate(results, 1):
    print("=" * 80)
    print("RESULT:", i)
    print("Source:", os.path.basename(doc.metadata.get("source", "")))
    print("Page:", doc.metadata.get("page"))
    print(doc.page_content[:500])

RESULT: 1
Source: ICICI_Home_Loan_Documents.pdf
Page: 0
APPLY FOR HOME LOAN
List of Documents required for Home Loan Based on profile
Individuals:
Identity proof and address proof of individuals:
Aadhaar or Passport or Driving License or Voter ID or National Rural Employment Guarantee Act (NREGA) Card or Letter from
the National Population Register (NPR) containing details of your name and address.
DOB Proof:
Passport/Driving License/Voter ID/NREGA Card/Letter from the NPR containing details of your name and address.
Non-Individuals (Entity):
Proof o
RESULT: 2
Source: SBI_HOME_LOANS_APPLICATION_FORM.pdf
Page: 0
Creative 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
Home Loan  
Application Form
RESULT: 3
Source: ICICI_Home_Loan_Documents.pdf
Page: 0
ICICI Bank - ICICI Home Loan Documents
Official Source: https://www.icici.bank.in/personal-banki

In [15]:
# Test embedding with English question
test_embedding_en = embeddings.embed_query(
    "Who is eligible for a home loan?"
)

print("English embedding dimension:", len(test_embedding_en))


# Test embedding with Marathi question
test_embedding_mr = embeddings.embed_query(
    "होम लोनसाठी कोण पात्र आहे?"
)

print("Marathi embedding dimension:", len(test_embedding_mr))


# Test embedding with Hindi question
test_embedding_hi = embeddings.embed_query(
    "होम लोन के लिए कौन पात्र है?"
)

print("Hindi embedding dimension:", len(test_embedding_hi))

English embedding dimension: 384
Marathi embedding dimension: 384
Hindi embedding dimension: 384


In [16]:
import os
import re


def rerank_loan_results(results, query, top_k=4):

    query_lower = query.lower()

    candidates = []

    # =====================================================
    # 1. QUERY-BASED KEYWORDS
    # =====================================================

    query_keywords = []

    keyword_groups = {
        "documents": [
            "document", "documents", "proof", "papers",
            "कागदपत्र", "दस्तऐवज",
            "दस्तावेज", "कागजात"
        ],

        "eligibility": [
            "eligible", "eligibility", "qualify",
            "पात्र", "पात्रता",
            "योग्य", "योग्यता"
        ],

        "kyc": [
            "kyc", "know your customer",
            "केवायसी", "केवाईसी"
        ],

        "interest": [
            "interest", "interest rate", "floating rate",
            "व्याज", "व्याजदर",
            "ब्याज", "ब्याज दर"
        ],

        "emi": [
            "emi", "equated monthly instalment",
            "equated monthly installment",
            "ईएमआय", "हप्ता",
            "किस्त"
        ],

        "repayment": [
            "repayment", "repay", "payment",
            "परतफेड", "परतफेडी",
            "भुगतान", "चुकौती"
        ],

        "prepayment": [
            "prepayment", "foreclosure", "pre-close",
            "pre closure", "pre-closure",
            "आगाऊ परतफेड",
            "फोरक्लोजर",
            "पूर्व भुगतान"
        ],

        "pmay": [
            "pmay", "pmay-u", "pmay-u 2.0",
            "प्रधानमंत्री आवास योजना",
            "प्रधानमंत्री आवास योजना",
            "आवास योजना"
        ],

        "grievance": [
            "complaint", "grievance", "ombudsman",
            "complaint redressal",
            "तक्रार", "तक्रार निवारण",
            "शिकायत", "शिकायत निवारण",
            "लोकपाल"
        ],

        "loan": [
            "home loan", "housing loan", "loan",
            "होम लोन", "गृह कर्ज",
            "गृह ऋण"
        ]
    }

    for group, keywords in keyword_groups.items():

        for keyword in keywords:

            if keyword in query_lower:

                query_keywords.append(group)
                break

    # =====================================================
    # 2. RERANK EACH RETRIEVED DOCUMENT
    # =====================================================

    for doc, score in results:

        content = doc.page_content.lower()

        boost = 0

        # -------------------------------------------------
        # Query keyword matching
        # -------------------------------------------------

        matched_groups = 0

        for group in query_keywords:

            for keyword in keyword_groups[group]:

                if keyword in content:
                    matched_groups += 1
                    break

        # More query-relevant groups → better score
        boost -= matched_groups * 0.20

        # -------------------------------------------------
        # Strong document-specific signals
        # -------------------------------------------------

        # Home loan documents
        if any(x in content for x in [
            "home loan documents",
            "documents required for home loan",
            "identity proof",
            "address proof",
            "income proof"
        ]):
            if "documents" in query_keywords:
                boost -= 0.50

        # Eligibility
        if any(x in content for x in [
            "eligibility criteria",
            "eligible borrower",
            "eligibility"
        ]):
            if "eligibility" in query_keywords:
                boost -= 0.50

        # KYC
        if any(x in content for x in [
            "know your customer",
            "kyc",
            "customer due diligence"
        ]):
            if "kyc" in query_keywords:
                boost -= 0.50

        # Interest
        if any(x in content for x in [
            "interest rate",
            "floating rate",
            "repo linked",
            "interest rates"
        ]):
            if "interest" in query_keywords:
                boost -= 0.50

        # EMI / repayment
        if any(x in content for x in [
            "equated monthly instalment",
            "equated monthly installment",
            "emi",
            "repayment"
        ]):
            if any(x in query_keywords for x in [
                "emi", "repayment"
            ]):
                boost -= 0.50

        # Prepayment / foreclosure
        if any(x in content for x in [
            "prepayment",
            "foreclosure",
            "pre-closure",
            "pre closure"
        ]):
            if "prepayment" in query_keywords:
                boost -= 0.60

        # PMAY
        if any(x in content for x in [
            "pmay-u",
            "pmay-u 2.0",
            "pradhan mantri awas yojana",
            "interest subsidy scheme",
            "beneficiary led construction"
        ]):
            if "pmay" in query_keywords:
                boost -= 0.70
            else:
                # Penalize PMAY documents for unrelated queries
                boost += 0.15

        # Grievance / Ombudsman
        if any(x in content for x in [
            "ombudsman",
            "grievance",
            "complaint",
            "complaint redressal"
        ]):
            if "grievance" in query_keywords:
                boost -= 0.70

        # -------------------------------------------------
        # Generic irrelevant-document penalty
        # -------------------------------------------------

        # If query is not PMAY, PMAY-heavy content gets a
        # small penalty.
        if "pmay" not in query_keywords:

            pmay_signals = [
                "pmay-u",
                "pmay-u 2.0",
                "pradhan mantri awas yojana",
                "beneficiary led construction"
            ]

            pmay_count = sum(
                content.count(signal)
                for signal in pmay_signals
            )

            if pmay_count >= 2:
                boost += 0.30

        # =================================================
        # FINAL SCORE
        # Lower = better
        # =================================================

        final_score = score + boost

        candidates.append(
            (doc, final_score, score)
        )

    # =====================================================
    # 3. SORT
    # =====================================================

    candidates.sort(key=lambda x: x[1])

    # =====================================================
    # 4. REMOVE DUPLICATE CONTENT
    # =====================================================

    unique = []
    seen = set()

    for doc, final_score, original_score in candidates:

        # Normalize whitespace
        text_key = re.sub(
            r"\s+",
            " ",
            doc.page_content.strip().lower()
        )

        if text_key not in seen:

            seen.add(text_key)

            unique.append(
                (doc, final_score, original_score)
            )

        if len(unique) >= top_k:
            break

    return unique

In [17]:
def expand_query(question):

    q = question.lower().strip()

    # =========================================================
    # HOME LOAN / HOUSING LOAN
    # =========================================================
    if (
        "home loan" in q
        or "housing loan" in q
        or "होम लोन" in q
        or "गृह कर्ज" in q
        or "गृह ऋण" in q
    ):
        return (
            question
            + " home loan housing loan "
              "home finance housing finance "
              "loan application borrower "
              "eligibility documents repayment"
        )

    # =========================================================
    # HOME LOAN ELIGIBILITY
    # =========================================================
    elif (
        "eligibility" in q
        or "eligible" in q
        or "पात्रता" in q
        or "पात्र" in q
        or "eligibility criteria" in q
    ):
        return (
            question
            + " home loan eligibility criteria "
              "eligible borrower applicant "
              "income age repayment capacity "
              "credit history loan eligibility"
        )

    # =========================================================
    # HOME LOAN DOCUMENTS
    # =========================================================
    elif (
        "document" in q
        or "documents" in q
        or "कागदपत्र" in q
        or "कागदपत्रे" in q
        or "दस्तावेज" in q
        or "दस्तऐवज" in q
        or "कागजात" in q
    ):
        return (
            question
            + " home loan documents "
              "documents required for home loan "
              "identity proof address proof "
              "income proof salary slips bank statements "
              "property documents KYC documents"
        )

    # =========================================================
    # KYC
    # =========================================================
    elif (
        "kyc" in q
        or "know your customer" in q
        or "केवायसी" in q
        or "केवाईसी" in q
    ):
        return (
            question
            + " KYC Know Your Customer "
              "customer due diligence identity verification "
              "identity proof address proof PAN Aadhaar "
              "KYC update periodic KYC"
        )

    # =========================================================
    # INTEREST RATE
    # =========================================================
    elif (
        "interest" in q
        or "interest rate" in q
        or "व्याज" in q
        or "व्याजदर" in q
        or "ब्याज" in q
        or "ब्याज दर" in q
        or "floating rate" in q
    ):
        return (
            question
            + " home loan interest rate "
              "interest rate floating rate fixed rate "
              "benchmark rate lending rate "
              "loan interest calculation"
        )

    # =========================================================
    # EMI
    # =========================================================
    elif (
        "emi" in q
        or "equated monthly" in q
        or "ईएमआय" in q
        or "हप्ता" in q
        or "किस्त" in q
    ):
        return (
            question
            + " EMI Equated Monthly Instalment "
              "home loan EMI principal interest "
              "monthly instalment repayment "
              "loan tenure EMI calculation"
        )

    # =========================================================
    # REPAYMENT
    # =========================================================
    elif (
        "repayment" in q
        or "repay" in q
        or "परतफेड" in q
        or "परतफेडी" in q
        or "भुगतान" in q
        or "चुकौती" in q
    ):
        return (
            question
            + " home loan repayment "
              "loan repayment schedule EMI "
              "principal interest instalments "
              "repayment tenure"
        )

    # =========================================================
    # PREPAYMENT / FORECLOSURE
    # =========================================================
    elif (
        "prepayment" in q
        or "pre-payment" in q
        or "foreclosure" in q
        or "pre closure" in q
        or "pre-closure" in q
        or "फोरक्लोजर" in q
        or "पूर्व भुगतान" in q
        or "आगाऊ परतफेड" in q
    ):
        return (
            question
            + " home loan prepayment foreclosure "
              "pre-closure loan closure before maturity "
              "prepayment charges foreclosure charges "
              "outstanding principal loan settlement"
        )

    # =========================================================
    # PMAY-U 2.0
    # =========================================================
    elif (
        "pmay" in q
        or "pmay-u" in q
        or "pmay u" in q
        or "pmay-u 2.0" in q
        or "pmay u 2.0" in q
        or "पीएमएवाई" in q
        or "प्रधानमंत्री आवास" in q
        or "आवास योजना" in q
    ):
        return (
            question
            + " PMAY-U 2.0 Pradhan Mantri Awas Yojana Urban "
              "housing scheme affordable housing "
              "beneficiary eligibility housing assistance "
              "Beneficiary Led Construction BLC "
              "Affordable Housing in Partnership AHP "
              "Interest Subsidy Scheme ISS"
        )

    # =========================================================
    # GRIEVANCE / COMPLAINT / OMBUDSMAN
    # =========================================================
    elif (
        "complaint" in q
        or "grievance" in q
        or "ombudsman" in q
        or "complaint redressal" in q
        or "तक्रार" in q
        or "तक्रार निवारण" in q
        or "शिकायत" in q
        or "शिकायत निवारण" in q
        or "लोकपाल" in q
    ):
        return (
            question
            + " loan complaint grievance redressal "
              "bank complaint customer complaint "
              "RBI Integrated Ombudsman Scheme "
              "complaint escalation complaint resolution "
              "banking grievance"
        )

    # =========================================================
    # PROPERTY DOCUMENTS
    # =========================================================
    elif (
        "property document" in q
        or "property documents" in q
        or "मालमत्ता कागदपत्र" in q
        or "मालमत्तेची कागदपत्रे" in q
        or "संपत्ति दस्तावेज" in q
    ):
        return (
            question
            + " home loan property documents "
              "title deed property papers "
              "sale deed ownership proof "
              "encumbrance certificate property tax receipts"
        )

    # =========================================================
    # SALARIED APPLICANT
    # =========================================================
    elif (
        "salaried" in q
        or "salary" in q
        or "पगारदार" in q
        or "नोकरी" in q
    ):
        return (
            question
            + " salaried home loan applicant "
              "salary slips income proof bank statements "
              "Form 16 employment proof income documents"
        )

    # =========================================================
    # SELF-EMPLOYED
    # =========================================================
    elif (
        "self employed" in q
        or "self-employed" in q
        or "व्यावसायिक" in q
        or "स्वयंरोजगार" in q
    ):
        return (
            question
            + " self employed home loan applicant "
              "business proof income proof ITR "
              "bank statements financial statements "
              "business stability documents"
        )

    # =========================================================
    # DEFAULT
    # =========================================================
    else:
        return question

In [18]:
def get_loan_sources(question):

    q = question.lower().strip()

    # =========================================================
    # 1. PMAY-U 2.0
    # =========================================================
    if (
        "pmay" in q
        or "pmay-u" in q
        or "pmay u" in q
        or "pmay-u 2.0" in q
        or "pmay u 2.0" in q
        or "पीएमएवाई" in q
        or "प्रधानमंत्री आवास" in q
        or "आवास योजना" in q
    ):
        return [
            "Operational-Guidelines-of-PMAY-U-2.pdf",
            "Operational-Guidelines-of-PMAY-U-2-Hindi.pdf"
        ]

    # =========================================================
    # 2. KYC
    # =========================================================
    if (
        "kyc" in q
        or "know your customer" in q
        or "केवायसी" in q
        or "केवाईसी" in q
    ):
        return [
            "RBI_mastery.pdf"
        ]

    # =========================================================
    # 3. LOAN INTEREST / EMI
    # =========================================================
    if (
        "interest" in q
        or "interest rate" in q
        or "floating rate" in q
        or "emi" in q
        or "repayment" in q
        or "व्याज" in q
        or "व्याजदर" in q
        or "ब्याज" in q
        or "ईएमआय" in q
        or "परतफेड" in q
        or "किस्त" in q
    ):
        return [
            "RBI-Intresti-on-loans.pdf"
        ]

    # =========================================================
    # 4. PREPAYMENT / FORECLOSURE
    # =========================================================
    if (
        "prepayment" in q
        or "pre-payment" in q
        or "foreclosure" in q
        or "pre closure" in q
        or "pre-closure" in q
        or "पूर्व भुगतान" in q
        or "फोरक्लोजर" in q
        or "आगाऊ परतफेड" in q
    ):
        return [
            "RBI-setlement.pdf"
        ]

    # =========================================================
    # 5. PENAL CHARGES
    # =========================================================
    if (
        "penal charge" in q
        or "penalty" in q
        or "penal charges" in q
        or "दंड" in q
        or "दंडात्मक शुल्क" in q
    ):
        return [
            "RBi-penalty-charges.pdf"
        ]

    # =========================================================
    # 6. PROPERTY DOCUMENT RELEASE
    # =========================================================
    if (
        "property document" in q
        or "property documents" in q
        or "release of property" in q
        or "title document" in q
        or "मालमत्ता कागदपत्र" in q
        or "मालमत्तेची कागदपत्रे" in q
    ):
        return [
            "RBI-Resposible-docs.pdf"
        ]

    # =========================================================
    # 7. GRIEVANCE / OMBUDSMAN
    # =========================================================
    if (
        "complaint" in q
        or "grievance" in q
        or "ombudsman" in q
        or "complaint redressal" in q
        or "तक्रार" in q
        or "तक्रार निवारण" in q
        or "शिकायत" in q
        or "शिकायत निवारण" in q
        or "लोकपाल" in q
    ):
        return [
            "Ombudsman_Scheme_English.pdf"
        ]

    # =========================================================
    # 8. ICICI HOME LOAN DOCUMENTS
    # =========================================================
    if (
        "document" in q
        or "documents" in q
        or "कागदपत्र" in q
        or "कागदपत्रे" in q
        or "दस्तावेज" in q
        or "दस्तऐवज" in q
        or "कागजात" in q
    ):
        return [
            "ICICI_Home_Loan_Documents.pdf",
            "ICICI_Home_Loan_FAQs.pdf",
            "SBI_HOME_LOANS_APPLICATION_FORM.pdf",
            "SBI_HOME_LOAN_MITC.pdf"
        ]

    # =========================================================
    # 9. SBI HOME LOAN
    # =========================================================
    if (
        "sbi" in q
        or "state bank" in q
    ):
        return [
            "SBI_HOME_LOANS_APPLICATION_FORM.pdf",
            "SBI_HOME_LOAN_MITC.pdf"
        ]

    # =========================================================
    # 10. GENERAL HOME LOAN
    # =========================================================
    if (
        "home loan" in q
        or "housing loan" in q
        or "होम लोन" in q
        or "गृह कर्ज" in q
        or "गृह ऋण" in q
    ):
        return [
            "ICICI_Home_Loan_Documents.pdf",
            "ICICI_Home_Loan_FAQs.pdf",
            "SBI_HOME_LOANS_APPLICATION_FORM.pdf",
            "SBI_HOME_LOAN_MITC.pdf"
        ]

    # =========================================================
    # 11. DEFAULT
    # =========================================================
    return None

In [19]:
def retrieve_documents(question, top_k=4):

    # =====================================================
    # STEP 1: Expand the user's query
    # =====================================================

    expanded_question = expand_query(question)

    # =====================================================
    # STEP 2: Detect relevant source documents
    # =====================================================

    allowed_sources = get_loan_sources(question)

    # =====================================================
    # STEP 3: Retrieve a larger candidate pool
    # =====================================================

    results = vectorstore.similarity_search_with_score(
        expanded_question,
        k=100
    )

    # =====================================================
    # STEP 4: Filter by source if a specific source
    #         category was detected
    # =====================================================

    if allowed_sources:

        filtered_results = []

        for doc, score in results:

            source = os.path.basename(
                doc.metadata.get("source", "")
            )

            if source in allowed_sources:
                filtered_results.append(
                    (doc, score)
                )

        # Only replace results if source filtering
        # actually returned something
        if filtered_results:
            results = filtered_results

    # =====================================================
    # STEP 5: RERANK RESULTS
    # =====================================================

    reranked_results = rerank_loan_results(
        results,
        question,
        top_k=top_k
    )

    # =====================================================
    # STEP 6: Return documents only
    # =====================================================

    final_documents = []

    for doc, final_score, original_score in reranked_results:

        final_documents.append(doc)

    return final_documents

In [20]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
You are LoanGuide AI, a multilingual RAG-based Home Loan and
Housing Finance information assistant.

Your job is to answer the user's question using ONLY the information
provided in the CONTEXT.

============================================================
IMPORTANT RULES
============================================================

1. Use ONLY the information available in the CONTEXT.

2. Do NOT use outside knowledge, memory, assumptions, or general
knowledge.

3. Do NOT invent loan rules, eligibility criteria, charges, interest
rates, documents, government benefits, or procedures.

4. If the answer is clearly available in the CONTEXT, answer the
question directly.

5. Use the most directly relevant passage from the CONTEXT.

6. Keep the answer focused on exactly what the user asked.

7. Do NOT add unnecessary information that is not required to answer
the question.

8. If multiple documents contain relevant information, use the
information that directly answers the question and cite the relevant
sources.

9. If the question is about a specific bank, use information from that
bank's documents when available.

10. Do NOT assume that a rule from one bank applies to another bank.

11. For RBI rules or regulatory information, rely only on the RBI
documents present in the CONTEXT.

12. For PMAY-U 2.0 questions, rely only on the PMAY-U 2.0 documents
present in the CONTEXT.

13. Do NOT mix information from unrelated documents just because the
words are similar.

============================================================
LANGUAGE RULES
============================================================

Answer in the SAME LANGUAGE as the user's question.

English question → English answer.

Marathi question → Marathi answer.

Hindi question → Hindi answer.

Mixed Marathi/English → simple Marathi.

Mixed Hindi/English → simple Hindi.

Use simple and easy-to-understand language.

============================================================
SOURCE AND PAGE CITATION RULES
============================================================

Every factual answer MUST include the source document and page number.

Use ONLY the source name and page number explicitly provided in the
CONTEXT.

NEVER invent a source.

NEVER invent a page number.

NEVER use a page number from memory.

NEVER cite a page that is not present in the CONTEXT.

If the CONTEXT contains:

[Source: ICICI_Home_Loan_Documents.pdf, Page: 2]

then cite exactly:

(Source: ICICI_Home_Loan_Documents.pdf, Page: 2)

For Marathi:

(स्रोत: ICICI_Home_Loan_Documents.pdf, पृष्ठ: 2)

For Hindi:

(स्रोत: ICICI_Home_Loan_Documents.pdf, पेज: 2)

============================================================
MULTIPLE SOURCES
============================================================

If the answer uses information from multiple documents, cite each
relevant source.

Example:

(Source: RBI_mastery.pdf, Page: 15)
(Source: SBI_HOME_LOAN_MITC.pdf, Page: 3)

Do NOT cite documents that were not used to answer the question.

============================================================
BANK-SPECIFIC INFORMATION
============================================================

If the user asks:

"ICICI Bank home loan documents"

answer using ICICI-related context.

If the user asks:

"SBI home loan documents"

answer using SBI-related context.

Do NOT combine ICICI and SBI requirements unless the user explicitly
asks for a comparison.

============================================================
GOVERNMENT SCHEME INFORMATION
============================================================

For PMAY-U 2.0 questions:

Use only the PMAY-U 2.0 information available in the CONTEXT.

Do not use unrelated home-loan information unless it directly answers
the question.

============================================================
KYC / RBI INFORMATION
============================================================

For KYC questions:

Use only the RBI/KYC information available in the CONTEXT.

Do not assume that a general KYC rule applies to every bank unless the
CONTEXT supports that statement.

============================================================
PREPAYMENT / FORECLOSURE
============================================================

For prepayment or foreclosure questions:

Answer only from the retrieved documents.

Do not invent or assume charges, exemptions, timelines, or conditions.

If the context contains bank-specific or loan-type-specific conditions,
clearly mention that they are specific to that source.

============================================================
FALLBACK RULE
============================================================

Use the following fallback ONLY when the CONTEXT genuinely does not
contain enough information to answer the question:

"I do not have enough information in the provided documents to answer
this question. Please consult the relevant bank, RBI, government
department, or authorized person for more clarity."

Marathi:

"दिलेल्या कागदपत्रांमध्ये या प्रश्नाचे उत्तर देण्यासाठी पुरेशी
माहिती उपलब्ध नाही. अधिक माहितीसाठी संबंधित बँक, RBI, सरकारी विभाग
किंवा अधिकृत व्यक्तीशी संपर्क साधा."

Hindi:

"दिए गए दस्तावेज़ों में इस प्रश्न का उत्तर देने के लिए पर्याप्त
जानकारी उपलब्ध नहीं है। अधिक जानकारी के लिए संबंधित बैंक, RBI,
सरकारी विभाग या अधिकृत व्यक्ति से संपर्क करें।"

IMPORTANT:

If the CONTEXT contains enough information to answer the question,
DO NOT use the fallback.

NEVER give an answer followed by the fallback.

The response must be either:

A) A complete answer with citation

OR

B) The fallback response.

Never both.

============================================================
CONTEXT
============================================================

{context}

============================================================
QUESTION
============================================================

{question}

============================================================
FINAL CHECK
============================================================

Before answering, verify:

- Did I answer only from the CONTEXT?
- Did I answer the actual question?
- Did I use the correct language?
- Did I use simple language?
- Did I use the exact source name?
- Did I use the exact page number shown in the CONTEXT?
- Did I avoid inventing information?
- Did I avoid mixing unrelated bank information?
- Did I avoid unnecessary details?
- Did I avoid giving a fallback when the answer is available?
- Did I avoid citing unused documents?

Now provide the final answer.
""")

In [21]:
# 3. Build the RAG chain (Replaces RetrievalQA)
# This chain: Fetches chunks -> Formats Prompt -> Sends to DeepSeek -> Cleans Output

from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
import os


# =========================================================
# FORMAT RETRIEVED DOCUMENTS
# =========================================================

def format_docs(docs):

    formatted_docs = []

    for doc in docs:

        source = os.path.basename(
            doc.metadata.get("source", "Unknown")
        )

        page = doc.metadata.get("page", "Unknown")

        # LangChain PDF page index is usually 0-based.
        # Convert to human-readable PDF page number.
        if isinstance(page, int):
            page = page + 1

        formatted_docs.append(
            f"[Source: {source}, Page: {page}]\n"
            f"{doc.page_content}"
        )

    return "\n\n---\n\n".join(formatted_docs)


# =========================================================
# CREATE RAG CHAIN
# =========================================================

rag_chain = (
    {
        "context": lambda question: format_docs(
            retrieve_documents(question)
        ),
        "question": RunnablePassthrough()
    }
    | prompt
    | chat_model
    | StrOutputParser()
)


print("✅ LoanGuide AI RAG chain created successfully!")

✅ LoanGuide AI RAG chain created successfully!


In [22]:
response = rag_chain.invoke(
    "What documents are required for a home loan?"
)

print(response)

The documents required for a home loan include:

**For Individuals:**
- Identity proof and address proof: Aadhaar, Passport, Driving License, Voter ID, NREGA Card, or Letter from the National Population Register (NPR) containing details of your name and address.  
- Date of Birth proof: Passport, Driving License, Voter ID, NREGA Card, or Letter from the NPR containing details of your name and address.  

**For Self-Employed Applicants:**
- IT returns, proof of business, recent bank statements, proof of identity, proof of address, and property documents.  

**For Non-Individuals (Entities):**
- Registration Certificate, Udyam Registration Certificate (URC), Certificate/License issued by municipal authorities under Shop and Establishment Act, Sales and Income Tax returns, CST/VAT/GST Certificate, Registration document issued by Sales Tax/Service Tax/Professional Tax authorities, or IEC (Importer Exporter Code).  

(Source: ICICI_Home_Loan_Documents.pdf, Page: 1)  
(Source: ICICI_Home_Loa

In [23]:
response = rag_chain.invoke(
    "होम लोन के लिए कौन से दस्तावेज आवश्यक हैं?"
)

print(response)

होम लोन के लिए आवश्यक दस्तावेज निम्नलिखित हैं:  

**व्यक्तियों के लिए:**  
1. पहचान प्रमाण और पता प्रमाण: आधार कार्ड, पासपोर्ट, ड्राइविंग लाइसेंस, वोटर आईडी, NREGA कार्ड या NPR से प्राप्त पत्र जिसमें आपका नाम और पता हो।  
2. जन्मतिथि प्रमाण: पासपोर्ट, ड्राइविंग लाइसेंस, वोटर आईडी, NREGA कार्ड या NPR से प्राप्त पत्र।  

**स्वरोजगार आवेदकों के लिए:**  
1. आयकर रिटर्न, व्यवसाय प्रमाण, हाल के बैंक स्टेटमेंट, पहचान प्रमाण, पता प्रमाण और संपत्ति संबंधी दस्तावेज।  

(स्रोत: ICICI_Home_Loan_Documents.pdf, पेज: 1)  
(स्रोत: ICICI_Home_Loan_Documents.pdf, पेज: 4)


In [24]:
response = rag_chain.invoke("होम लोनसाठी कोणती कागदपत्रे आवश्यक आहेत?")

print(response)

होम लोनसाठी खालील कागदपत्रे आवश्यक आहेत:  

1. ओळखपत्र (Identity Proof): आधार कार्ड, पासपोर्ट, ड्रायव्हिंग लायसन्स, मतदार ओळखपत्र, NREGA कार्ड किंवा NPR पत्र.  
2. पत्ता पुरावा (Address Proof): आधार कार्ड, पासपोर्ट, ड्रायव्हिंग लायसन्स, मतदार ओळखपत्र, NREGA कार्ड किंवा NPR पत्र.  
3. जन्मतारीख पुरावा (DOB Proof): पासपोर्ट, ड्रायव्हिंग लायसन्स, मतदार ओळखपत्र, NREGA कार्ड किंवा NPR पत्र.  
4. उत्पन्न पुरावा (Income Proof): पगारी व्यक्तींसाठी पगारपट्ट्या आणि बँक स्टेटमेंट्स, स्वयंरोजगार व्यक्तींसाठी IT रिटर्न्स, व्यवसाय पुरावा आणि बँक स्टेटमेंट्स.  
5. मालमत्ता संबंधित कागदपत्रे (Property Documents): मालमत्तेशी संबंधित कागदपत्रे.  

(स्रोत: ICICI_Home_Loan_Documents.pdf, पृष्ठ: 1)  
(स्रोत: ICICI_Home_Loan_Documents.pdf, पृष्ठ: 3)


In [25]:
response = rag_chain.invoke("Why is KYC required for a home loan?")

print(response)

KYC (Know Your Customer) is required for a home loan to ensure compliance with regulatory requirements and to verify the identity of the customer. This helps in preventing fraud, money laundering, and other financial crimes. The RBI mandates that banks carry out Customer Due Diligence (CDD) as part of the KYC process for all financial transactions, including home loans.  

(Source: RBI_mastery.pdf, Page: 27)  
(Source: RBI_mastery.pdf, Page: 63)


In [26]:
response = rag_chain.invoke("What is prepayment of a home loan?")

print(response)

Prepayment of a home loan refers to paying off a part or the entire loan amount before the scheduled tenure ends. For floating-rate home loans with ICICI Bank, there are no charges for foreclosure or part-prepayment. However, for fixed-rate home loans, charges may apply if the loan is not closed with your own funds.  

(Source: ICICI_Home_Loan_FAQs.pdf, Page: 10)


In [28]:
response = rag_chain.invoke("What is PMAY-U 2.0?")

print(response)

PMAY-U 2.0, or Pradhan Mantri Awas Yojana - Urban 2.0, is a scheme designed to address the affordable housing requirement in urban areas. It includes four verticals: Beneficiary Led Construction (BLC), Affordable Housing in Partnership (AHP), Affordable Rental Housing (ARH), and Interest Subsidy Scheme (ISS). The scheme aims to improve the quality of life by incorporating global best practices and outcomes from consultations with various stakeholders like States/UTs, Banks, Housing Finance Companies, and the Private Sector.  

(Source: Operational-Guidelines-of-PMAY-U-2.pdf, Page: 5)


In [29]:
response = rag_chain.invoke("स्वरोजगार करने वाले व्यक्ति को होम लोन के लिए कौन से दस्तावेज चाहिए?")

print(response)

स्वरोजगार करने वाले व्यक्ति को ICICI बँकेतर्फे होम लोनसाठी खालील दस्तावेज आवश्यक आहेत:  
- आयकर परतावा (IT returns)  
- व्यवसायाचा पुरावा  
- अलीकडील बँक स्टेटमेंट्स  
- ओळखपत्राचा पुरावा  
- पत्त्याचा पुरावा  
- मालमत्तेसंबंधित दस्तावेज  

(स्रोत: ICICI_Home_Loan_Documents.pdf, पृष्ठ: 4)


In [30]:
response = rag_chain.invoke("होम लोनसाठी पात्रता कशी ठरवली जाते?")

print(response)

होम लोनसाठी पात्रता वय, उत्पन्न, विद्यमान दायित्वे, क्रेडिट स्कोअर आणि मालमत्तेच्या मूल्य यासारख्या घटकांवर आधारित ठरवली जाते. (स्रोत: ICICI_Home_Loan_FAQs.pdf, पृष्ठ: 2)


In [31]:
response = rag_chain.invoke("होम लोन की ब्याज दर से संबंधित जानकारी क्या है?")

print(response)

होम लोन की ब्याज दर आवेदक के क्रेडिट स्कोर, लोन राशि और आवेदित होम लोन के प्रकार पर निर्भर करती है। ये कारक प्रत्येक व्यक्ति की वित्तीय प्रोफाइल और लोन आवश्यकताओं के अनुसार ब्याज दर निर्धारित करने में मदद करते हैं।  

(स्रोत: ICICI_Home_Loan_Documents.pdf, पेज: 3)


In [32]:
response = rag_chain.invoke("अगर बैंक मेरी होम लोन की शिकायत का समाधान नहीं करता है तो मैं आगे क्या कर सकता हूँ?")

print(response)

अगर बैंक आपकी होम लोन की शिकायत का समाधान नहीं करता है, तो आप ऑनलाइन पोर्टल (https://cms.rbi.org.in) के माध्यम से शिकायत दर्ज कर सकते हैं। आप इलेक्ट्रॉनिक या फिजिकल मोड में भी शिकायत जमा कर सकते हैं। फिजिकल शिकायत को आपके या आपके अधिकृत प्रतिनिधि द्वारा हस्ताक्षरित होना चाहिए। शिकायत को रिजर्व बैंक द्वारा निर्धारित फॉर्मेट और जानकारी के साथ जमा करना होगा।  

(स्रोत: Ombudsman_Scheme_English.pdf, पेज: 8)


# Hallucination Test

In [33]:
response = rag_chain.invoke("What is the government subsidy amount under the National Urban Home Loan Benefit Scheme 2026?")

print(response)

I do not have enough information in the provided documents to answer this question. Please consult the relevant bank, RBI, government department, or authorized person for more clarity.


In [34]:
response = rag_chain.invoke("What documents are required to get a personal loan for buying a car?")

print(response)

I do not have enough information in the provided documents to answer this question. Please consult the relevant bank, RBI, government department, or authorized person for more clarity.


In [47]:
response = rag_chain.invoke(
    "What is Solar System"
)

print(response)

The provided CONTEXT does not contain any information about the "Solar System." Therefore, I cannot answer this question based on the available documents.

Fallback response:  
"I do not have enough information in the provided documents to answer this question. Please consult the relevant bank, RBI, government department, or authorized person for more clarity."


In [ ]:
# Saving the Model

In [40]:
# Save FAISS vector database

faiss_folder = "faiss_index"

vectorstore.save_local(faiss_folder)

print("✅ FAISS index saved successfully!")
print("Saved folder:", faiss_folder)

✅ FAISS index saved successfully!
Saved folder: faiss_index


In [41]:
# verify faiss save or not
import os

print("FAISS folder exists:", os.path.exists("faiss_index"))
print("Files:", os.listdir("faiss_index"))

FAISS folder exists: True
Files: ['index.faiss', 'index.pkl']


In [43]:
import pandas as pd